# Implementação do Modelo - Fase 2

Reogarnização dos arquivos do projeto para que reflitam as fases aplicadas na pesquisa. Conforme Guia_Fase2_Passo2_Estrutura_Modulos.cd

In [6]:
%load_ext autoreload
%autoreload 2


import random

from philder import infra
from philder import database_consolidation as consolidacao
from philder import data_processing as processamento
from philder import database_balance as balanceamento
from philder import performance_assessment as avaliacao


# --- Preparação ---
infra.criar_estrutura_diretorios()

# --- Módulo A: consolidação da base ---
#brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
#brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")
#df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

# --- Módulo B: processamento de texto ---
# df = processamento.pipeline_processamento(df)

# --- Módulo C: balanceamento + treino + teste ---
# df_balanceado = balanceamento.aplicar_undersampling(df)
# df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)
# tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
# dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
# trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
# predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
# balanceamento.salvar_modelo(trainer, tokenizer)

# --- Módulo D: avaliação de desempenho ---
# metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
# eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
# avaliacao.gerar_curva_perda(trainer.state.log_history)
# avaliacao.salvar_resultados({**metricas, **eficiencia})


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Estrutura de diretórios criada/confirmada:
  - dados_philder/
  - modelo/  (com modelo\checkpoints/)
  - resultados/


# A) DATABASE CONSOLIDATION

<DESENVOLVER>

In [7]:
# --- Módulo A: consolidação da base ---
brutos_legitimos = consolidacao.importar_corpus_spamassassin("dados_philder/spamassassin")
brutos_phishing = consolidacao.importar_corpus_nazario("dados_philder/Nazario.csv")

# 19/09/2026 -  Ajuste da base para refletir a quantidade de amostras do artigo original, especifcamente
# da base do phishtank
# Nazario tem mais e-mails de phishing do que o PhishTank usado no artigo (501) —
# reduz por amostragem aleatória (sem reposição) para equiparar a mesma quantidade.
# SpamAssassin já está na mesma contagem do artigo (2.501), não precisa de ajuste.
## 25/09/2026 desativado pois foi incorporada na etapa de deduplicação
## random.seed(42)
## brutos_phishing = random.sample(brutos_phishing, 501)

legitimos_amostrados, phishing_amostrados = consolidacao.amostrar_quantidade_artigo(
    brutos_legitimos, brutos_phishing, deduplicar_antes=True
)

df = consolidacao.consolidar_dataset(legitimos_amostrados, phishing_amostrados)

## 25/09/2026 ajuste por implementação da deduplicação
## df = consolidacao.consolidar_dataset(brutos_legitimos, brutos_phishing)

Total de e-mails legítimos carregados: 2501
Total de e-mails de phishing carregados: 1565 (coluna: 'body')
Pré-processamento concluído: 1565 registros (38 ficaram vazios)
Deduplicação prévia do pool de 'phishing' (texto processado): 1565 e-mails -> 1472 únicos (93 duplicatas descartadas antes da amostragem)
Pré-processamento concluído: 2501 registros (2 ficaram vazios)
Deduplicação prévia do pool de 'legítimos' (texto processado): 2501 e-mails -> 2446 únicos (55 duplicatas descartadas antes da amostragem)
Aviso: pool único de 'legítimos' tem 2446 e-mails, igual ou abaixo do alvo do artigo (2501). Mantendo todos os 2446 disponíveis.
Amostragem para escala do artigo: 2446 legítimos + 501 phishing = 2947 e-mails (alvo do artigo: 2501+501=3002) [deduplicar_antes=True]
Dataset consolidado: 2947 registros — legítimos: 2446, phishing: 501


# B) DATA PROCESSING
<DISCORRER MAIS>

In [8]:
# --- Módulo B: processamento de texto ---
df = processamento.pipeline_processamento(df)


Pré-processamento concluído: 2947 registros (1 ficaram vazios)


# C) DATABASE BALANCE

In [9]:
# --- Módulo C: balanceamento + treino + teste ---
df_balanceado = balanceamento.aplicar_undersampling(df)
df_treino, df_teste = balanceamento.dividir_treino_teste(df_balanceado)

####### Utilizar para avaliar amostragem utilizada
#############inicio debug##############################
#dup_no_balanceado = df_balanceado.duplicated(subset="body_processado", keep=False)
#print("duplicatas no balanceado:", dup_no_balanceado.sum(), "de", len(df_balanceado))
#print(df_balanceado["class_label"].value_counts())

#dup_legit = df_balanceado[df_balanceado["class_label"] == 0].duplicated(subset="body_processado", keep=False)
#dup_phish = df_balanceado[df_balanceado["class_label"] == 1].duplicated(subset="body_processado", keep=False)
#print("duplicatas DENTRO de legítimos:", dup_legit.sum())
#print("duplicatas DENTRO de phishing:", dup_phish.sum())

#sobreposicao = set(df_treino["body_processado"]) & set(df_teste["body_processado"])
#print(f"E-mails processados idênticos entre treino e teste: {len(sobreposicao)}")

#if sobreposicao:
#    afetados_treino = df_treino[df_treino["body_processado"].isin(sobreposicao)]
#    afetados_teste = df_teste[df_teste["body_processado"].isin(sobreposicao)]
#    print("treino:", len(afetados_treino), afetados_treino["class_label"].value_counts().to_dict())
#    print("teste:", len(afetados_teste), afetados_teste["class_label"].value_counts().to_dict())

#raise SystemExit("Checagem de duplicatas concluída — parando aqui antes do treino.")

#######################fim debug###################

tokenizer, modelo_tb = balanceamento.carregar_tokenizer_e_modelo()
dataset_treino, dataset_teste = balanceamento.tokenizar_dataset(df_treino, df_teste, tokenizer)
trainer, tempo_treino = balanceamento.treinar_modelo(modelo_tb, tokenizer, dataset_treino, dataset_teste)
predicoes, tempo_teste = balanceamento.testar_modelo(trainer, dataset_teste)
balanceamento.salvar_modelo(trainer, tokenizer)

Balanceamento aplicado: {1: 501, 0: 501}
Treino: {1: 351, 0: 350}
Teste: {0: 151, 1: 150}
duplicatas no balanceado: 0 de 1002
class_label
1    501
0    501
Name: count, dtype: int64
duplicatas DENTRO de legítimos: 0
duplicatas DENTRO de phishing: 0
E-mails processados idênticos entre treino e teste: 0


SystemExit: Checagem de duplicatas concluída — parando aqui antes do treino.

\\?\C:\Users\alvar\AppData\Roaming\jupyterlab-desktop\envs\env_philder1\Lib\site-packages\IPython\core\interactiveshell.py:3783: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# D) PERFORMANCE ASSESSMENT
## Alterações identificadas:

- [x] Incluir na tabela de métricas o Tempo de treino e o tempo de teste e tempo de detecção
- [x] Equacionar o número de épocas com o do artigo (10 épocas)
- [x] Executar um treinamento com o dataset atual, simulando a quantidade de amostras do artigo original
- [x] Deduplicar dados de dataset de phishing para melhorar qualidade do trainamento e teste
- [x] Deduplicar dados de dataset de phishing de de ham para comparar resultado em relação a deduplicação somente de phishing. 
- [ ] Identificar padrão de unidades usada no artigo para avaliar ajuste e necessidade de conversão de dados, pois os resultados são bem diferentes. Considerar também o hardware utilizado
- [ ]  Finalizado ajustes rodar notebook no google colabs e comparar os resultados IES, TES, RTDE


# Resultados

<details>
<summary>Clique para expandir</summary>

## Resultados do artigo original:

### Tabela 1 — Métricas de Desempenho Preditivo (PHILDER = TinyBERT (2), execução com undersampling)

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| TinyBERT (2) | 0,9927 | 0,9896 | 0,9960 | 0,9927 | 111,83 | 4,48 | 14,8841 |

### Tabela 2 — Métricas de Eficiência Computacional (PHILDER = TinyBERT + undersampling, valores brutos dos gráficos)
**Valores não normalizados**
| Métrica | Valor bruto |
|---|---|
| TES | 0,0389 |
| IES | 1,0788 |
| RTDE | 324,7750 |

## Resultados deste Trabalho Fase2

### Tabela 1 — Métricas de Desempenho Preditivo

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Modelo | Acc. | Prec. | Rec. | F1 |
|---|---|---|---|---|
| philder |0,9936 | 0,9957 | 0,9915 | 0,9936 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**
| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9867 | 1,0 | 0,9733 | 0,9865 | 255.8025 | 4.5329 | 15.0595 |

**Data Medição: 21/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/ evitar overfiting**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9933 | 1,0 | 0,9867 | 0,9933 | 205,5063 | 2,1591 | 7,1734 |


**Data Medição: 22/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras `df.drop_duplicates`**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9792 | 0,97992 | 0,9792 | 0,9792 | 298,1438 | 2,0680 | 7,1556 |


**Data Medição: 23/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9900 | 0,9868 | 0,9933 | 0,9900 | 263,5980 | 2,3067 | 7,6633 |


**Data Medição: 24/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing e Legítimas `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

A precisão volta a ficar suspeita (~1,0). Investigação revela dois bugs reais: (1) a célula do Módulo A no notebook nunca chamava amostrar_quantidade_artigo — usava código residual de 19/09 (random.sample sem deduplicação)


**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9939 | 0,9989 | 0,9989 | 0,9939 | 198,06 | 2,23 | 7,40 |
|Desv Padrão | 0,0033 | 0,0027 | 0,0054 | 0,0033 | 25,02 | 0,16 | 0,55 |

**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** e **Legítimas** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Modelo | Acc. | Prec. | Rec. | F1 | T. Treino (S)| T. Teste (s) | Tempo de Detecção (ms) |
|---|---|---|---|---|---|---|---|
| philder |0,9967 | 0,9978 | 0,9956 | 0,9967 | 352,24 | 3,92 | 13,03 |
|Desv Padrão | 0,0021 | 0,0034 | 0,0034 | 0,0021 | 44,90 | 0,29|  0,96 |


### Tabela 2 — Métricas de Eficiência Computacional

**Data Medição: 17/09/2026**

**Base Nazario 1501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,00268 |
| IES | 0,1474 |
| RTDE | 138,3831 |

**Data Medição: 19/09/2026**

**Base Nazario 501 amostras**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0039 |
| IES | 0,2177 |
| RTDE | 65,505 |

**Data Medição: 21/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

| Métrica | Valor bruto |
|---|---|
| TES | 0,0048 |
| IES | 0,4601 |
| RTDE | 138,4772 |

**Data Medição: 22/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de todas as mensagens (ham, phishing) `df.drop_duplicates` DEPOIS DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO** 

| Métrica | Valor bruto |
|---|---|
| TES | 0,0033 |
| IES | 0,4735 |
| RTDE | 136,8492 |

**Data Medição: 23/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

| Métrica | Valor bruto |
|---|---|
| TES | 0,00378|
| IES | 0,4292 |
| RTDE | 129,1910 |


**Data Medição: 24/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de deduplicação das amostras de mensagens phishing e Legítimas `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

A precisão volta a ficar suspeita (~1,0). Investigação revela dois bugs reais: (1) a célula do Módulo A no notebook nunca chamava amostrar_quantidade_artigo — usava código residual de 19/09 (random.sample sem deduplicação)

**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Métrica | Valor bruto |
|---|---|
| TES | 0,005088 ± 0,000664 |
| IES | 0,447961 ± 0,032449 |
| RTDE | 134,8363 ± 9,7670 |

**Data Medição: 25/09/2026**

**Base Nazario 501 amostras**

**Até 10 épocas, + parametros p/  evitar overfiting**

**Aplicação de técnica de **deduplicação** das amostras de mensagens **phishing** e **ham** `df.drop_duplicates` ANTES DA CONSOLIDAÇÃO DA BASE PARA TREINAMENTO**

**Média 6 rodadas de trainamento**

| Métrica | Valor bruto |
|---|---|
| TES | 0,002872 ± 0,000406 |
| IES | 0,255279 ± 0,018430 |
| RTDE | 76,8516 ± 5,5475 |

</details>

## Análise Crítica da Avaliação dos Resultados

- Até o momento os resultados de Eficiência são da ordem de 10x piores que o do artigo, indicando que as etapas de treino e testes foram executadoas em um ambiente de poder computacional elevado. O fator mais relevante para a discrepância nos valores foi o tempo. Avaliar este ponto com mais atenção e garantir que as métricas estão realizando o calculo da forma adequada,
- Verificado que, após a equalização da base com o artigo, foi identificado que algumas amostras identicas no dataset treinado incfluenciaram na precisão do modelo (100%). Para contornar a situação foi realizado um tratamento no dataset para a normalização das amostras duplicadas `df.drop_duplicates` e realizada nova medição com este dataset normalizado. Após execução do treinamento o treinamento se mostrou mais consistente,e os dados duplicados causaram interferência nos resultados anteriores.
- Após a ajuste dos dados duplicados, houve mais um ajuste neste sentido, para retirar os dados duplicados da base de phishing (Nazario) antes da consolidação, fazendo, dessa forma, que a base para trainamento e teste tenha 501 amostras consistentes, onde é esperado uma melhora da perforamnce do treino e teste.
    - Foi necessário realizar um debug de código após resultados inconsistêntes de acurácia. Após ajuste os dados passaram a ser consistentes e foram executadas várias rodadas de trinamento (**resultado_final_25092026_estavel[i].csv**)
- A deduplicação foi incluída também na amostra de ham (spamassassin) para verificar se os resultados seriam mais consistentes. Como a técnica do undersample foi aplicada, a deduplicação não influiu na quantidade de amostras usadas no treino e no teste, mas as filtrou, impedindo que amostras duplicadas após o processo de normalzação do corpo do email (rem cabeçalho, links etc..) não fizessem parte do corpo de treino. O resultado consolidado da execução deste processo segue detalhado abaixo:
   
  ```
    Total de e-mails legítimos carregados: 2501
    Total de e-mails de phishing carregados: 1565 (coluna: 'body')
    Pré-processamento concluído: 1565 registros (38 ficaram vazios)
    Deduplicação prévia do pool de 'phishing' (texto processado): 1565 e-mails -> 1472 únicos (93 duplicatas descartadas antes da amostragem)
    Pré-processamento concluído: 2501 registros (2 ficaram vazios)
    Deduplicação prévia do pool de 'legítimos' (texto processado): 2501 e-mails -> 2446 únicos (55 duplicatas descartadas antes da amostragem)
    Aviso: pool único de 'legítimos' tem 2446 e-mails, igual ou abaixo do alvo do artigo (2501). Mantendo todos os 2446 disponíveis.
    Amostragem para escala do artigo: 2446 legítimos + 501 phishing = 2947 e-mails (alvo do artigo: 2501+501=3002) [deduplicar_antes=True]
    Dataset consolidado: 2947 registros — legítimos: 2446, phishing: 501
  ```
  
  - O treinamento foi executado em 6 rodadas novamente (**resultado_final_25092026-3_estavel[i].csv**). 
- Aparentemente os valores estão normalizados (0,1). Avaliar qual técnica foi utilizada (normalização?) para os vaores de métrica de desempenho.
- 

In [ ]:
# --- Módulo D: avaliação de desempenho ---
metricas = avaliacao.calcular_metricas_primarias(df_teste["class_label"].tolist(), predicoes)
eficiencia = avaliacao.calcular_metricas_eficiencia(metricas["accuracy"], tempo_treino, tempo_teste, len(df_teste))
avaliacao.gerar_curva_perda(trainer.state.log_history)
avaliacao.salvar_resultados({**metricas, **eficiencia})